# 07 — Unification & nettoyage → table canonique

**Rôle :** fusionner House + Sénat, mapper au schéma unifié, rattacher l'identité (**BioGuideID** via le référentiel), normaliser tickers/dates/montants, **dédupliquer** (SHA-256 de la clé naturelle), documenter chaque règle, sauvegarder (CSV + parquet).

**Prérequis :** notebooks 01, 04, 05 exécutés · `pip install pyarrow`.

**Cellule 1 — Imports & chemins.**

In [ ]:
import json, hashlib, re
from pathlib import Path
import pandas as pd

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() or (p / ".git").exists():
            return p
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
PROC_HOUSE   = PROJECT_ROOT / "data" / "processed" / "house"
PROC_SENATE  = PROJECT_ROOT / "data" / "processed" / "senate"
PROCESSED    = PROJECT_ROOT / "data" / "processed"
REFERENCE    = PROJECT_ROOT / "data" / "reference" 

**Cellule 2 — Chargement des deux moitiés + référentiel.**

In [ ]:
def load(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()
house  = load(PROC_HOUSE / "house_transactions.csv")
senate = load(PROC_SENATE / "senate_transactions.csv")
ref    = pd.read_csv(REFERENCE / "legislators.csv")
print("House:", len(house), "| Sénat:", len(senate), "| Référentiel:", len(ref))

**Cellule 3 — Schéma unifié.** On force les mêmes colonnes des deux côtés.

In [ ]:
SCHEMA = ["source","provenance","bioguide_id","declarant_name","chamber","party",
          "state_district","committee_membership","committees_key_flag",
          "transaction_date","disclosure_date","ticker","asset_description","asset_type",
          "operation_type","amount_range","amount_midpoint","owner","doc_id","source_url"]

def conform(df):
    df = df.copy()
    for c in SCHEMA:
        if c not in df.columns:
            df[c] = None
    return df[SCHEMA]

combined = pd.concat([conform(house), conform(senate)], ignore_index=True)
print("Combiné :", len(combined))

**Cellule 4 — Réconciliation d'identité.** Le Sénat (notebook 05) fournit déjà le `bioguide_id` ; on ne rattache **par nom** que les lignes sans identifiant (House), via un index `nom normalisé → BioGuideID`.

In [ ]:
def norm(name):
    if not isinstance(name, str):
        return ""
    s = name.lower()
    s = re.sub(r"\b(hon|mr|mrs|ms|dr|jr|sr|iii|ii)\b", "", s)
    s = re.sub(r"[^a-z ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

name_to_bio = {}
for _, r in ref.iterrows():
    variants = json.loads(r["name_variants"]) if isinstance(r["name_variants"], str) else []
    for v in list(variants) + [r["declarant_name"]]:
        name_to_bio[norm(v)] = r["bioguide_id"]

# Le Sénat fournit déjà bioguide_id ; on ne rattache par nom QUE les lignes sans identifiant (House).
if "bioguide_id" not in combined.columns:
    combined["bioguide_id"] = None
provided = combined["bioguide_id"].notna()
miss = ~provided
combined.loc[miss, "bioguide_id"] = combined.loc[miss, "declarant_name"].map(lambda n: name_to_bio.get(norm(n)))
print("Fournis :", int(provided.sum()),
      "| rattachés par nom :", int(combined.loc[miss, "bioguide_id"].notna().sum()),
      "| total :", int(combined["bioguide_id"].notna().sum()), "/", len(combined))

**Cellule 5 — Chambre + enrichissement.** `chamber` vient de la **source** (fiable, pas de NaN) ; parti et commissions viennent du référentiel via BioGuideID.

In [ ]:
# chamber depuis la SOURCE (évite les NaN sur les lignes non rattachées)
combined["chamber"] = combined["source"].map({"house": "house", "senate": "senate"}).fillna(combined["chamber"])
ref_idx = ref.drop_duplicates("bioguide_id").set_index("bioguide_id")
for c in ["party", "committee_membership", "committees_key_flag"]:
    if c in ref_idx.columns:
        combined[c] = combined["bioguide_id"].map(ref_idx[c])

**Cellule 6 — Normalisations documentées.** R1 : tickers majuscules, `--`→null. R2 : dates ISO. R3 : `amount_midpoint` depuis la fourchette si absent.

In [ ]:
# R1 — tickers
combined["ticker"] = combined["ticker"].astype("string").str.upper().replace({"--": pd.NA, "": pd.NA})
# R2 — dates
for c in ["transaction_date", "disclosure_date"]:
    combined[c] = pd.to_datetime(combined[c], errors="coerce").dt.date
# R3 — midpoint depuis amount_range si manquant
def midpoint(rng):
    if not isinstance(rng, str):
        return None
    nums = [int(x.replace(",", "")) for x in re.findall(r"[\d,]+", rng) if x.replace(",", "").isdigit()]
    return sum(nums) / len(nums) if nums else None
mask = combined["amount_midpoint"].isna() & combined["amount_range"].notna()
combined.loc[mask, "amount_midpoint"] = combined.loc[mask, "amount_range"].map(midpoint)

**Cellule 7 — Déduplication déterministe.** Clé = chambre+déclarant+date+ticker+type+montant → SHA-256 ; une ligne par clé.

In [ ]:
KEY = ["chamber", "declarant_name", "transaction_date", "ticker", "operation_type", "amount_range"]
def natural_key(r):
    return hashlib.sha256("|".join(str(r.get(k, "")) for k in KEY).encode()).hexdigest()
combined["natural_key_hash"] = combined.apply(natural_key, axis=1)
before = len(combined)
combined = combined.drop_duplicates("natural_key_hash").reset_index(drop=True)
print(f"Dédup : {before} -> {len(combined)}")

**Cellule 8 — Sauvegarde de la table canonique** (CSV lisible + parquet robuste).

In [ ]:
combined.to_csv(PROCESSED / "congress_transactions.csv", index=False)
try:
    combined.to_parquet(PROCESSED / "congress_transactions.parquet", index=False)
    print("Écrit CSV + parquet :", len(combined), "lignes")
except Exception as e:
    print("CSV écrit. Parquet ignoré (pip install pyarrow) :", e)

Table canonique prête ✅ — validation **`08_validation_quiver`**.